In [8]:
import torch
import lightning.pytorch as pl
from pytorch_forecasting import TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss
from lightning.pytorch.callbacks import (
    EarlyStopping,
    LearningRateMonitor,
    ModelCheckpoint,
)
from lightning.pytorch.loggers import CSVLogger
from pathlib import Path
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

In [9]:
DATA_DIR = Path('../data')

In [10]:
train = torch.load(f'{DATA_DIR}/bin/training_temporaldataset.pkl', weights_only=False)
val = torch.load(f'{DATA_DIR}/bin/validation_temporaldataset.pkl', weights_only=False)
test = torch.load(f'{DATA_DIR}/bin/test_temporaldataset.pkl', weights_only=False)

In [11]:
# ==================== HYPERPARAMETERS ====================
# Model Architecture
HIDDEN_SIZE = 32  # Main model width (16-64 for crypto)
ATTENTION_HEAD_SIZE = 4  # Number of attention heads
HIDDEN_CONTINUOUS_SIZE = 16  # Size for continuous variable processing
LSTM_LAYERS = 2  # Number of LSTM layers in encoder/decoder
DROPOUT = 0.2  # Dropout rate (0.1-0.3, higher for noisy crypto)

# Training
LEARNING_RATE = 1e-3  # Initial learning rate
BATCH_SIZE = 64  # Batch size (smaller for volatile data)
MAX_EPOCHS = 100  # Maximum training epochs
GRADIENT_CLIP_VAL = 0.1  # Gradient clipping threshold

# Early Stopping
PATIENCE = 15  # Epochs without improvement before stopping
MIN_DELTA = 1e-4  # Minimum change to qualify as improvement

# DataLoader
NUM_WORKERS = 0  # DataLoader workers (0 for notebook stability)

# Loss Function (QuantileLoss for uncertainty estimation)
QUANTILES = [0.1, 0.5, 0.9]  # 10th, 50th (median), 90th percentiles

print(f'Hyperparameters:')
print(
  f'  Model: hidden={HIDDEN_SIZE}, heads={ATTENTION_HEAD_SIZE}, lstm_layers={LSTM_LAYERS}, dropout={DROPOUT}'
)
print(f'  Training: lr={LEARNING_RATE}, batch={BATCH_SIZE}, epochs={MAX_EPOCHS}')
print(f'  Early stopping: patience={PATIENCE}, min_delta={MIN_DELTA}')

Hyperparameters:
  Model: hidden=32, heads=4, lstm_layers=2, dropout=0.2
  Training: lr=0.001, batch=64, epochs=100
  Early stopping: patience=15, min_delta=0.0001


In [12]:
# Create DataLoaders
train_dataloader = train.to_dataloader(
  train=True, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS
)
val_dataloader = val.to_dataloader(
  train=False,
  batch_size=BATCH_SIZE * 4,  # Larger batch for validation (faster)
  num_workers=NUM_WORKERS,
)

print(f'Train batches: {len(train_dataloader):,}')
print(f'Val batches: {len(val_dataloader):,}')

Train batches: 8,000
Val batches: 419


In [13]:
# Initialize TFT Model
tft = TemporalFusionTransformer.from_dataset(
  train,
  hidden_size=HIDDEN_SIZE,
  attention_head_size=ATTENTION_HEAD_SIZE,
  hidden_continuous_size=HIDDEN_CONTINUOUS_SIZE,
  lstm_layers=LSTM_LAYERS,
  dropout=DROPOUT,
  learning_rate=LEARNING_RATE,
  loss=QuantileLoss(quantiles=QUANTILES),
  optimizer='adam',
  reduce_on_plateau_patience=4,
)

print(f'Model parameters: {tft.size() / 1e6:.2f}M')
print(f'Loss function: QuantileLoss with quantiles {QUANTILES}')

Model parameters: 0.13M
Loss function: QuantileLoss with quantiles [0.1, 0.5, 0.9]


/Users/matt/Documents/Algo-Trading/env/lib/python3.13/site-packages/lightning/pytorch/utilities/parsing.py:210: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/Users/matt/Documents/Algo-Trading/env/lib/python3.13/site-packages/lightning/pytorch/utilities/parsing.py:210: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


In [14]:
# Setup Trainer with Callbacks
early_stop = EarlyStopping(
  monitor='val_loss', patience=PATIENCE, min_delta=MIN_DELTA, mode='min', verbose=True
)

lr_monitor = LearningRateMonitor(logging_interval='epoch')

checkpoint = ModelCheckpoint(
  monitor='val_loss',
  mode='min',
  save_top_k=1,
  filename='tft-{epoch:02d}-{val_loss:.4f}',
)

# CSV Logger (fixes tensorboard warning)
logger = CSVLogger('lightning_logs', name='tft_training')

trainer = pl.Trainer(
  max_epochs=MAX_EPOCHS,
  gradient_clip_val=GRADIENT_CLIP_VAL,
  callbacks=[early_stop, lr_monitor, checkpoint],
  logger=logger,
  enable_progress_bar=True,
  log_every_n_steps=50,
)

print(
  f'Trainer configured with max_epochs={MAX_EPOCHS}, gradient_clip={GRADIENT_CLIP_VAL}'
)
print(f'Logs will be saved to: {logger.log_dir}')

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Trainer configured with max_epochs=100, gradient_clip=0.1
Logs will be saved to: lightning_logs/tft_training/version_0


# Training

**Note**: Training will use MPS (Apple Silicon GPU) if available. If you encounter errors, you can disable GPU by changing `accelerator="cpu"` in the Trainer.


In [15]:
# Train Model
trainer.fit(tft, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

print(f'\nTraining complete!')
print(f'Best model saved at: {checkpoint.best_model_path}')


   | Name                               | Type                            | Params | Mode 
------------------------------------------------------------------------------------------------
0  | loss                               | QuantileLoss                    | 0      | train
1  | logging_metrics                    | ModuleList                      | 0      | train
2  | input_embeddings                   | MultiEmbedding                  | 152    | train
3  | prescalers                         | ModuleDict                      | 640    | train
4  | static_variable_selection          | VariableSelectionNetwork        | 3.9 K  | train
5  | encoder_variable_selection         | VariableSelectionNetwork        | 39.7 K | train
6  | decoder_variable_selection         | VariableSelectionNetwork        | 20.6 K | train
7  | static_context_variable_selection  | GatedResidualNetwork            | 4.3 K  | train
8  | static_context_initial_hidden_lstm | GatedResidualNetwork            | 4.3 K  

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/Users/matt/Documents/Algo-Trading/env/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=9` in the `DataLoader` to improve performance.
/Users/matt/Documents/Algo-Trading/env/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=9` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

/Users/matt/Documents/Algo-Trading/env/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# Load Best Model
best_model_path = checkpoint.best_model_path
best_tft = TemporalFusionTransformer.load_from_checkpoint(best_model_path)

print(f'Loaded best model from: {best_model_path}')

# Testing


In [ ]:
# Create Test DataLoader
test_dataloader = test.to_dataloader(
  train=False, batch_size=BATCH_SIZE * 4, num_workers=NUM_WORKERS
)

print(f'Test batches: {len(test_dataloader):,}')

In [ ]:
# Generate Predictions
predictions = best_tft.predict(
  test_dataloader, return_y=True, trainer_kwargs=dict(accelerator='cpu')
)

# predictions is a tuple: (predictions_tensor, actuals_tensor)
# For QuantileLoss: predictions has shape [n_samples, n_quantiles]
pred_tensor = predictions.output
actuals = predictions.y[0]  # Shape: [n_samples, 1]

print(f'Predictions shape: {pred_tensor.shape}')
print(f'Actuals shape: {actuals.shape}')

In [ ]:
# Calculate Metrics
# Extract quantile predictions: [q10, q50 (median), q90]
pred_np = pred_tensor.cpu().numpy()
actual_np = actuals.cpu().numpy().flatten()

# Use median (q50) for point prediction metrics
median_idx = QUANTILES.index(0.5)
pred_median = pred_np[:, median_idx]

# Point prediction metrics
mae = mean_absolute_error(actual_np, pred_median)
rmse = np.sqrt(mean_squared_error(actual_np, pred_median))
r2 = r2_score(actual_np, pred_median)

# Coverage: % of actuals within prediction interval [q10, q90]
q10_idx = QUANTILES.index(0.1)
q90_idx = QUANTILES.index(0.9)
pred_q10 = pred_np[:, q10_idx]
pred_q90 = pred_np[:, q90_idx]
coverage = np.mean((actual_np >= pred_q10) & (actual_np <= pred_q90)) * 100

print('=' * 50)
print('TEST SET METRICS')
print('=' * 50)
print(f'MAE:      {mae:.6f}')
print(f'RMSE:     {rmse:.6f}')
print(f'R²:       {r2:.4f}')
print(f'Coverage: {coverage:.1f}% (target: 80% for [0.1, 0.9] interval)')
print('=' * 50)

In [ ]:
# Visualization: Sample predictions with uncertainty bands
n_samples = 200  # Show first N predictions

fig, ax = plt.subplots(figsize=(14, 5))

x = np.arange(n_samples)
ax.plot(x, actual_np[:n_samples], label='Actual', color='black', linewidth=1.5)
ax.plot(
  x, pred_median[:n_samples], label='Predicted (median)', color='blue', linewidth=1
)
ax.fill_between(
  x,
  pred_q10[:n_samples],
  pred_q90[:n_samples],
  alpha=0.3,
  color='blue',
  label='80% prediction interval',
)

ax.set_xlabel('Sample')
ax.set_ylabel('next_close_log (normalized)')
ax.set_title('TFT Predictions vs Actuals (Test Set Sample)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()